Mini Project: Dry Bean Type Classification

1. Import and Load the Data
•	Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn etc.)
•	Load the dataset and explore it using .head(), .info(), and .describe()


In [ ]:
# Importing all Libraries
import numpy as npy
import pandas as pds
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as mat
import seaborn as sbn
import plotly.express as px
import plotly.graph_objects as go

import warnings
import missingno
warnings.filterwarnings("ignore")
%matplotlib inline
sbn.set(style="darkgrid",font_scale=1.5)
pds.set_option("display.max.columns",None)
pds.set_option("display.max.rows",None)

In [ ]:
# Importing all ML algorithms / metrices

get_ipython().system('pip install catboost')

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate


1. Import and Load the Data
•	Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn etc.)
•	Load the dataset and explore it using .head(), .info(), and .describe()


In [ ]:
# Dry_Beans_Multiclass_Data = pds.read_csv("Dry_Beans_Multiclass_Classification.csv")

file_path_1 = 'C:/Users/richi/DATA SCIENCE - IIT GUWAHATI/PRACTICAL/MINI PROJECT/DATA SETS/Dry_Beans_Multiclass_Classification.csv'
Dry_Beans_Multiclass_Data = pds.read_csv(file_path_1)


In [ ]:
# Visualizing Basic Information of Imported dataset.

print("First 5 rows of the DataFrame:")
print(Dry_Beans_Multiclass_Data.head())

print("\nDataFrame Information:")
print(Dry_Beans_Multiclass_Data.info())

print("\nDescriptive Statistics:")
print(Dry_Beans_Multiclass_Data.describe())

print("\nShape of the DataFrame:")
print(Dry_Beans_Multiclass_Data.shape)

2. Exploratory Data Analysis (EDA)
•	Visualize distributions of features using histograms and boxplots
•	Analyze the class distribution (check for class imbalance)
•	Plot feature correlations (eg heatmap)
•	Visualize multivariate relationships (pairplot)
•	Summarize key findings


Visualize distributions of features using histograms and boxplots

In [ ]:
# Identify numerical columns for distribution plots
numerical_cols = Dry_Beans_Multiclass_Data.select_dtypes(include=npy.number).columns

for col in numerical_cols:
    print(f"\n--- Distribution for {col} ---")
    fig, axes = mat.subplots(1, 2, figsize=(16, 6))
    sbn.histplot(Dry_Beans_Multiclass_Data[col], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Histogram of {col}')
    sbn.boxplot(y=Dry_Beans_Multiclass_Data[col], ax=axes[1], color='lightcoral')
    axes[1].set_title(f'Boxplot of {col}')
    mat.tight_layout()
    mat.show()

Analyze the class distribution (check for class imbalance)

In [ ]:
# Analyze the distribution of the target variable 'Class'
print("Class Distribution:")
print(Dry_Beans_Multiclass_Data['Class'].value_counts())

mat.figure(figsize=(10, 6))
sbn.countplot(data=Dry_Beans_Multiclass_Data, x='Class', palette='viridis')
mat.title('Distribution of Dry Bean Classes')
mat.xlabel('Bean Class')
mat.ylabel('Count')
mat.xticks(rotation=45)
mat.show()

print("\nClass Distribution (Percentages):")
print(Dry_Beans_Multiclass_Data['Class'].value_counts(normalize=True) * 100)

Plot feature correlations (eg heatmap)

In [ ]:
# Calculate the correlation matrix
correlation_matrix = Dry_Beans_Multiclass_Data.select_dtypes(include=npy.number).corr()

mat.figure(figsize=(16, 12))
sbn.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
mat.title('Feature Correlation Heatmap')
mat.show()

 Visualize multivariate relationships (pairplot)

Selecting a few key features for demonstration, including the target 'Class'
If we wants all numerical features, this can be adjusted. But executing that will take time. Therefore we are demonstrating with only Area, Perimeter, MajorAxisLength, MinorAxisLength, Class


In [ ]:
# For better visualization, we might select a subset of features for the pairplot
# or if the dataset is not too large, plot all numerical features.
# Given the number of features, let's include the target 'Class' to see relationships.

# features_for_pairplot = Dry_Beans_Multiclass_Data.columns.tolist()
features_for_pairplot = ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'Class']
# Create the pairplot
sbn.pairplot(Dry_Beans_Multiclass_Data[features_for_pairplot], hue='Class', palette='tab10')
mat.suptitle('Pairplot of Selected Features by Class', y=1.02) # Adjust suptitle position
mat.show()

Summarize Key Findings from EDA

Key Findings from EDA:

Feature Distributions :
    Histograms and boxplots revealed that many numerical features exhibit varying distributions, with some appearing skewed (e.g., Area, Perimeter, MajorAxisLength). Boxplots also indicated the presence of outliers in several features, which needs treatment in next step.

Class Distribution :
    The dataset shows a noticeable class imbalance. 'DERMASON' is the most frequent class, accounting for approximately 26.05% of the data, while 'BOMBAY' is the least frequent, representing only about 3.83%. This imbalance should be considered during model training to avoid bias towards the majority classes.

Feature Correlations :
    The correlation heatmap highlighted strong positive correlations among many of the geometric features (e.g., Area with Perimeter, MajorAxisLength, MinorAxisLength, ConvexArea, EquivDiameter). This high multicollinearity suggests that some features might convey similar information, and dimensionality reduction or careful feature selection might be beneficial for modeling.

Multivariate Relationships (Pairplot) :
    The pairplot, focusing on key features like Area, Perimeter,MajorAxisLength, and MinorAxisLength colored by 'Class', indicated that different bean types show varying degrees of separability based on these feature pairs. Some classes appear more distinct, while others overlap, suggesting that a combination of features will be crucial for accurate classification.

3. Missing Values & Outlier Treatment
•	Check for and handle missing values
•	Detect and treat outliers if needed (Z-score / IQR methods/boxplots)


Checking for Missing Values

In [ ]:
print("Missing values per column:")
print(Dry_Beans_Multiclass_Data.isnull().sum())
if Dry_Beans_Multiclass_Data.isnull().sum().sum() == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing values found in the dataset. Handling missing values is crucial.")

As we have seen in Box Plot distribution, there are multiple columns which has outliers.
Therefore, We need to treat them. For outlier detection, we will use the Interquartile Range (IQR) method. For treatment, we will cap these outliers to these upper and lower bounds to mitigate their impact without removing data points, as removal can lead to loss of information, especially with skewed data.

In [ ]:
# Select only numerical columns for outlier treatment
numerical_cols = Dry_Beans_Multiclass_Data.select_dtypes(include=npy.number).columns

# Create a copy to store treated data
Dry_Beans_Treated_Data = Dry_Beans_Multiclass_Data.copy()

print("Applying IQR-based outlier treatment (capping) to numerical features...")

for column in numerical_cols:
    Q1 = Dry_Beans_Treated_Data[column].quantile(0.25)
    Q3 = Dry_Beans_Treated_Data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap outliers
    Dry_Beans_Treated_Data[column] = npy.where(
        Dry_Beans_Treated_Data[column] < lower_bound,
        lower_bound,
        npy.where(
            Dry_Beans_Treated_Data[column] > upper_bound,
            upper_bound,
            Dry_Beans_Treated_Data[column]
        )
    )
    # Check how many values were capped in this column
    capped_count = ((Dry_Beans_Multiclass_Data[column] < lower_bound) | (Dry_Beans_Multiclass_Data[column] > upper_bound)).sum()
    if capped_count > 0:
        print(f"  - Capped {capped_count} outliers in column: {column}")

print("Outlier treatment complete. Displaying first 5 rows of the treated data:")
display(Dry_Beans_Treated_Data.head())

4. Feature Engineering & Preprocessing
•	Scale numerical features (StandardScaler / MinMaxScaler)
•	Encode categorical variables if necessary
•	Check and treat skewness if required
•	Split data into train/test sets (use stratified sampling while splitting)


#### Scaling Numerical Features and Encoding Target Variable

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split


# Separate features (X) and target (y)
X = Dry_Beans_Treated_Data.drop('Class', axis=1)
y = Dry_Beans_Treated_Data['Class']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=npy.number).columns
categorical_cols = X.select_dtypes(include='object').columns

# Initialize StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to numerical features
X_scaled = X.copy()
X_scaled[numerical_cols] = scaler.fit_transform(X_scaled[numerical_cols])

# Initialize LabelEncoder for the target variable
label_encoder = LabelEncoder()

# Encode the target variable
y_encoded = label_encoder.fit_transform(y)

print("Numerical features scaled and target variable encoded.")
print("First 5 rows of scaled features (X_scaled):")
display(X_scaled.head())
print("First 5 encoded target values (y_encoded):")
display(y_encoded[:5])
print("Original class names:", label_encoder.classes_)

Checking and Treating Skewness :
    Let's check the skewness of the numerical features after scaling. If highly skewed, we might apply transformations like log transform to make the distributions more symmetrical, which can improve model performance for some algorithms.

In [ ]:
# Calculate skewness for numerical features after scaling
skewness = X_scaled[numerical_cols].skew()
print("Skewness of numerical features after scaling:")
display(skewness.sort_values(ascending=False))

# Define a skewness threshold (e.g., 0.75)
skewness_threshold = 0.75
highly_skewed_cols = skewness[abs(skewness) > skewness_threshold].index

print(f"\nHighly skewed columns (absolute skewness > {skewness_threshold}):")
if len(highly_skewed_cols) > 0:
    print(highly_skewed_cols.tolist())
    print("Preparing to apply PowerTransformer to these columns.")
else:
    print("No highly skewed columns found that require transformation.")

Applying power transformation on Highly skewed columns.

In [ ]:
from sklearn.preprocessing import PowerTransformer

# Initialize PowerTransformer with Yeo-Johnson method
powertransformer = PowerTransformer(method='yeo-johnson', standardize=False) # standardize=False to only transform skewness

# Apply PowerTransformer to the highly skewed numerical columns in X_scaled
# Create a copy of X_scaled to avoid SettingWithCopyWarning
X_transformed = X_scaled.copy()

if len(highly_skewed_cols) > 0:
    # Fit and transform only the highly skewed columns
    X_transformed[highly_skewed_cols] = powertransformer.fit_transform(X_transformed[highly_skewed_cols])
    print("PowerTransformer (Yeo-Johnson) applied to highly skewed columns.")
    print("\nNew skewness for transformed columns:")
    display(X_transformed[highly_skewed_cols].skew().sort_values(ascending=False))
else:
    print("No highly skewed columns to transform.")

# Update X_scaled to reflect the transformation for subsequent steps
X_scaled = X_transformed

Splitting Data into Train/Test Sets :
    Finally, we will split the preprocessed data into training and testing sets. It's crucial to use stratified sampling because our target variable ('Class') shows class imbalance, and stratified sampling ensures that the proportion of each class is roughly the same in both the training and testing sets.

In [ ]:
# Split data into training and testing sets with stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Data split into training and testing sets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nClass distribution in original data (y):")
display(y.value_counts(normalize=True))
print("\nClass distribution in training set (y_train):")
display(pds.Series(y_train).value_counts(normalize=True).sort_index())
print("\nClass distribution in testing set (y_test):")
display(pds.Series(y_test).value_counts(normalize=True).sort_index())

5. Model Building: Try Multiple Classifiers
Train and test the dataset on a variety of supervised classification algorithms:
•	Logistic Regression
•	Decision Tree Classifier
•	Random Forest Classifier
•	K-Nearest Neighbors (KNN)
•	Support Vector Machine (SVM)
•	Ensemble Learning Methods
•	Naive Bayes and more…
Use Cross validation techniques to check if model performance is improved.


Model Training and Evaluation

In [ ]:
# Define a dictionary of models to train
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear', multi_class='ovr'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Support Vector Machine': SVC(random_state=42, probability=True), # probability=True for ROC AUC
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, auto_class_weights='Balanced') # Handle class imbalance
}

# Prepare lists to store results
results = []
model_names = []

# Define evaluation metrics for cross-validation
scoring = {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro'
}

print("Models and evaluation setup complete.")

Training Models with Stratified K-Fold Cross-Validation

In [ ]:
n_splits = 5 # Number of folds for cross-validation
strat_k_fold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n--- Training and evaluating {name} ---")
    model_names.append(name)

    # Perform cross-validation
    cv_results = cross_validate(model, X_train, y_train, cv=strat_k_fold, scoring=scoring, n_jobs=-1)

    # Calculate mean and std of each metric
    accuracy_mean = cv_results['test_accuracy'].mean()
    precision_mean = cv_results['test_precision_macro'].mean()
    recall_mean = cv_results['test_recall_macro'].mean()
    f1_mean = cv_results['test_f1_macro'].mean()

    accuracy_std = cv_results['test_accuracy'].std()
    precision_std = cv_results['test_precision_macro'].std()
    recall_std = cv_results['test_recall_macro'].std()
    f1_std = cv_results['test_f1_macro'].std()

    results.append({
        'Model': name,
        'Accuracy (Mean)': accuracy_mean,
        'Accuracy (Std)': accuracy_std,
        'Precision (Mean)': precision_mean,
        'Precision (Std)': precision_std,
        'Recall (Mean)': recall_mean,
        'Recall (Std)': recall_std,
        'F1-Score (Mean)': f1_mean,
        'F1-Score (Std)': f1_std
    })

    print(f"{name} - Mean Accuracy: {accuracy_mean:.4f} (+/- {accuracy_std:.4f})")
    print(f"{name} - Mean Precision: {precision_mean:.4f} (+/- {precision_std:.4f})")
    print(f"{name} - Mean Recall: {recall_mean:.4f} (+/- {recall_std:.4f})")
    print(f"{name} - Mean F1-Score: {f1_mean:.4f} (+/- {f1_std:.4f})")



In [ ]:
# Display the results in a DataFrame
results_df = pd.DataFrame(results)
# results_df = results_df.sort_values(by='Accuracy (Mean)', ascending=False).reset_index(drop=True)
results_df = results_df.sort_values(by=['Accuracy (Mean)', 'Accuracy (Std)'],ascending=[False, True]).reset_index(drop=True)
print("\n--- Model Cross-Validation Results ---")
display(results_df)

Best Model Selection and Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Select the best model (CatBoost Classifier based on cross-validation results)
best_model_name = results_df.iloc[1]['Model']
best_model = models[best_model_name]

print(f"Selected Best Model: {best_model_name}")

# Train the best model on the full training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
print("\n--- Model Evaluation on Test Set ---")
print(f"Accuracy on Test Set: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
mat.figure(figsize=(10, 8))
sbn.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
mat.title('Confusion Matrix for Best Model')
mat.xlabel('Predicted Label')
mat.ylabel('True Label')
mat.show()

6. Handling Class Imbalance
•	Apply techniques like:
o	SMOTE (Synthetic Minority Over-sampling)
o	Random Oversampling / Undersampling
o	Class weighting
•	Evaluate if performance improves on minority classes


We will use SMOTE (Synthetic Minority Over-sampling Technique) to address the class imbalance observed earlier. SMOTE generates synthetic samples for the minority class(es), which can help improve the performance of classifiers on these classes.

In [ ]:
from imblearn.over_sampling import SMOTE

print("Original class distribution in training set (y_train):")
display(pds.Series(y_train).value_counts().sort_index())

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE (y_train_resampled):")
display(pds.Series(y_train_resampled).value_counts().sort_index())

Now, let's retrain the models using the resampled training data (X_train_resampled, y_train_resampled) and evaluate their performance with cross-validation.

In [ ]:
# Prepare lists to store SMOTE results
smote_results = []

for name, model in models.items():
    print(f"\n--- Training and evaluating {name} with SMOTE ---")

    # Perform cross-validation on resampled data
    cv_results_smote = cross_validate(model, X_train_resampled, y_train_resampled, cv=strat_k_fold, scoring=scoring, n_jobs=-1)

    # Calculate mean and std of each metric
    accuracy_mean_smote = cv_results_smote['test_accuracy'].mean()
    precision_mean_smote = cv_results_smote['test_precision_macro'].mean()
    recall_mean_smote = cv_results_smote['test_recall_macro'].mean()
    f1_mean_smote = cv_results_smote['test_f1_macro'].mean()

    accuracy_std_smote = cv_results_smote['test_accuracy'].std()
    precision_std_smote = cv_results_smote['test_precision_macro'].std()
    recall_std_smote = cv_results_smote['test_recall_macro'].std()
    f1_std_smote = cv_results_smote['test_f1_macro'].std()

    smote_results.append({
        'Model': name,
        'Accuracy (Mean)': accuracy_mean_smote,
        'Accuracy (Std)': accuracy_std_smote,
        'Precision (Mean)': precision_mean_smote,
        'Precision (Std)': precision_std_smote,
        'Recall (Mean)': recall_mean_smote,
        'Recall (Std)': recall_std_smote,
        'F1-Score (Mean)': f1_mean_smote,
        'F1-Score (Std)': f1_std_smote
    })

    print(f"{name} with SMOTE - Mean Accuracy: {accuracy_mean_smote:.4f} (+/- {accuracy_std_smote:.4f})")
    print(f"{name} with SMOTE - Mean Precision: {precision_mean_smote:.4f} (+/- {precision_std_smote:.4f})")
    print(f"{name} with SMOTE - Mean Recall: {recall_mean_smote:.4f} (+/- {recall_std_smote:.4f})")
    print(f"{name} with SMOTE - Mean F1-Score: {f1_mean_smote:.4f} (+/- {f1_std_smote:.4f})")

In [ ]:
# Display the SMOTE results in a DataFrame
smote_results_df = pd.DataFrame(smote_results)
smote_results_df = smote_results_df.sort_values(by=['Accuracy (Mean)', 'Accuracy (Std)'],ascending=[False, True]).reset_index(drop=True)
print("\n--- Model Cross-Validation Results with SMOTE ---")
display(smote_results_df)

print("\n--- Original Model Cross-Validation Results ---")
display(results_df)

### Best Model Selection and Evaluation on Test Set (After SMOTE)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Select the best model from SMOTE results (LightGBM Classifier)
best_smote_model_name = smote_results_df.iloc[0]['Model']
best_smote_model = models[best_smote_model_name]

print(f"Selected Best Model after SMOTE: {best_smote_model_name}")

# Train the best SMOTE model on the full resampled training data
best_smote_model.fit(X_train_resampled, y_train_resampled)

# Make predictions on the original test set
y_pred_smote = best_smote_model.predict(X_test)

# Evaluate the model
print("\n--- Model Evaluation on Test Set (After SMOTE) ---")
print(f"Accuracy on Test Set (After SMOTE): {accuracy_score(y_test, y_pred_smote):.4f}")

print("\nClassification Report (After SMOTE):")
print(classification_report(y_test, y_pred_smote, target_names=label_encoder.classes_))

# Confusion Matrix
cm_smote = confusion_matrix(y_test, y_pred_smote)
mat.figure(figsize=(10, 8))
sbn.heatmap(cm_smote, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
mat.title(f'Confusion Matrix for {best_smote_model_name} (After SMOTE)')
mat.xlabel('Predicted Label')
mat.ylabel('True Label')
mat.show()

print("\n--- Comparison with Original Best Model (CatBoost) ---")
print("Original Best Model (CatBoost) Accuracy on Test Set: 0.9229") # From previous execution of cell 9c8689c2

7. Model Evaluation & Overfitting Check
Use appropriate classification metrics:
•	Accuracy
•	Precision, Recall, and F1-Score (for each class)
•	Confusion Matrix

Check for Overfitting:
Compare training vs test accuracy and performance metrics.


Model Evaluation & Overfitting Check :

Based on the cross-validation and test set evaluations, here's a summary of the performance of the models, especially focusing on the impact of SMOTE and checking for overfitting.

1. Evaluation of Models (without SMOTE) :

    From the initial cross-validation, the CatBoost model emerged as the best performer, achieving a mean accuracy of 0.9300 on the training data. On the unseen test set, this model achieved an accuracy of 0.9229. And Standard Deviation of accuracy is 0.003997, which is minimal , means our model is more stable.

    Metrics (CatBoost on Test Set):
    1. Accuracy: 0.9229
    2. Precision (Macro Avg): 0.94
    3. Recall (Macro Avg): 0.94
    4. F1-Score (Macro Avg): 0.94

2. Evaluation of Models (with SMOTE):

    After applying SMOTE to the training data to address class imbalance, the models were re-trained and evaluated using cross-validation. LightGBM showed the highest mean accuracy of 0.9602 on the resampled training data with minimal Variation of 0.003498.

    Metrics (LightGBM with SMOTE on Test Set) :
    1. Accuracy: 0.9207
    2. Precision (Macro Avg): 0.93
    3. Recall (Macro Avg): 0.93
    4. F1-Score (Macro Avg): 0.93

3. Overfitting Check:

To check for overfitting, we compare the cross-validation (training) performance with the test set performance.

CatBoost (without SMOTE):
    CV Accuracy (Mean): 0.9300
    Test Accuracy: 0.9229
    
Observation: The test accuracy is slightly lower than the CV accuracy, which is expected and indicates good generalization without significant overfitting. The difference is minimal, suggesting the model is not overly complex for the data.

LightGBM (with SMOTE):
    CV Accuracy (Mean): 0.9602
    Test Accuracy: 0.9207
    
Observation: There's a more noticeable drop from the CV accuracy to the test accuracy (from 0.9602 to 0.9207). This suggests that while SMOTE significantly improved performance on the balanced training folds (leading to higher CV scores), it might have caused the model to learn patterns specific to the synthetic samples, leading to a slight decrease in performance on the original, imbalanced test set. However, it's crucial to note that SMOTE generally improves recall and F1-score for minority classes, which is important for imbalanced datasets.

Conclusion on Overfitting:

Both models show reasonable generalization, but the LightGBM model trained with SMOTE exhibits a larger gap between its cross-validation and test performance. This could indicate a minor degree of overfitting to the synthetically generated data or simply that the original CatBoost model was more robust to the inherent class imbalance of the test set. Despite this, SMOTE successfully balanced the training data, and the macro-averaged metrics (Precision, Recall, F1-Score) generally improved or remained competitive, indicating better handling of all classes, particularly the minority ones.

8. Hyperparameter Tuning
•	Use GridSearchCV or RandomizedSearchCV to optimize parameters for top-performing models
•	Document the best parameters and performance improvement


We will use RandomizedSearchCV to find the optimal hyperparameters for the LightGBM model, which performed best on the SMOTE-resampled data. This approach is more computationally efficient than GridSearchCV for exploring a wide range of parameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# LightGBM Classifier (with SMOTE) - Hyperparameter Tuning
print("\n--- Tuning LightGBM with RandomizedSearchCV (after SMOTE) ---")

lgbm_model = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1) # verbose=-1 to suppress warnings

# Define the hyperparameter search space for LightGBM
lgbm_param_dist = {
    'n_estimators': [100, 200, 300, 500, 700],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'num_leaves': [20, 31, 40, 50, 60],
    'max_depth': [-1, 5, 8, 10, 12],
    'min_child_samples': [10, 20, 30, 40],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1, 2],
    'reg_lambda': [0, 0.1, 0.5, 1, 2],
}

# Initialize RandomizedSearchCV
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=lgbm_param_dist,
    n_iter=50,  # Number of parameter settings that are sampled. More is better but takes longer.
    scoring='accuracy',
    cv=strat_k_fold, # Use the same stratified k-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1  # Use all available cores
)

# Fit RandomizedSearchCV to the SMOTE-resampled training data
random_search_lgbm.fit(X_train_resampled, y_train_resampled)

print("\nBest parameters for LightGBM:", random_search_lgbm.best_params_)
print("Best cross-validation accuracy for LightGBM:", random_search_lgbm.best_score_)

# Store the best LightGBM model
best_lgbm_smote = random_search_lgbm.best_estimator_


Evaluate Tuned LightGBM on Test Set (after SMOTE)

In [ ]:
# Make predictions on the original test set with the tuned LightGBM model
y_pred_tuned_lgbm = best_lgbm_smote.predict(X_test)

print("\n--- Tuned LightGBM Model Evaluation on Test Set (After SMOTE) ---")
print(f"Accuracy on Test Set (Tuned LightGBM): {accuracy_score(y_test, y_pred_tuned_lgbm):.4f}")

print("\nClassification Report (Tuned LightGBM):")
print(classification_report(y_test, y_pred_tuned_lgbm, target_names=label_encoder.classes_))

# Confusion Matrix for Tuned LightGBM
cm_tuned_lgbm = confusion_matrix(y_test, y_pred_tuned_lgbm)
mat.figure(figsize=(10, 8))
sbn.heatmap(cm_tuned_lgbm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
mat.title('Confusion Matrix for Tuned LightGBM (After SMOTE)')
mat.xlabel('Predicted Label')
mat.ylabel('True Label')
mat.show()


Now, we will tune the CatBoost Classifier, which performed best without SMOTE, using RandomizedSearchCV on the original training data.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# CatBoost Classifier (without SMOTE) - Hyperparameter Tuning
print("\n--- Tuning CatBoost with RandomizedSearchCV (without SMOTE) ---")

# Ensure CatBoost is initialized without auto_class_weights for the non-SMOTE case
cat_model = CatBoostClassifier(random_state=42, verbose=0, auto_class_weights=None)

# Define the hyperparameter search space for CatBoost
cat_param_dist = {
    'iterations': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'depth': [4, 6, 8, 10],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'border_count': [32, 64, 128]
    # 'subsample': [0.6, 0.8, 1.0] # Removed as it's incompatible with default 'Bayesian' bootstrap_type
}

# Initialize RandomizedSearchCV
random_search_cat = RandomizedSearchCV(
    estimator=cat_model,
    param_distributions=cat_param_dist,
    n_iter=50,  # Number of parameter settings that are sampled
    scoring='accuracy',
    cv=strat_k_fold, # Use the same stratified k-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV to the original training data
random_search_cat.fit(X_train, y_train)

print("\nBest parameters for CatBoost:", random_search_cat.best_params_)
print("Best cross-validation accuracy for CatBoost:", random_search_cat.best_score_)

# Store the best CatBoost model
best_cat_no_smote = random_search_cat.best_estimator_

### Evaluate Tuned CatBoost on Test Set (without SMOTE)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Make predictions on the original test set with the tuned CatBoost model
y_pred_tuned_cat = best_cat_no_smote.predict(X_test)

print("\n--- Tuned CatBoost Model Evaluation on Test Set (Without SMOTE) ---")
print(f"Accuracy on Test Set (Tuned CatBoost): {accuracy_score(y_test, y_pred_tuned_cat):.4f}")

print("\nClassification Report (Tuned CatBoost):")
print(classification_report(y_test, y_pred_tuned_cat, target_names=label_encoder.classes_))

# Confusion Matrix for Tuned CatBoost
cm_tuned_cat = confusion_matrix(y_test, y_pred_tuned_cat)
mat.figure(figsize=(10, 8))
sbn.heatmap(cm_tuned_cat, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
mat.title('Confusion Matrix for Tuned CatBoost (Without SMOTE)')
mat.xlabel('Predicted Label')
mat.ylabel('True Label')
mat.show()


## 8. Hyperparameter Tuning Summary and Performance Improvement

### Tuned LightGBM (with SMOTE)

After applying `RandomizedSearchCV` to the LightGBM model trained on SMOTE-resampled data, the following optimal parameters were identified:

*   **Best Parameters:** `{'subsample': 0.8, 'reg_lambda': 0.1, 'reg_alpha': 0, 'num_leaves': 31, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 12, 'learning_rate': 0.1, 'colsample_bytree': 1.0}`
*   **Best Cross-Validation Accuracy (on resampled training data):** 0.9612
*   **Test Set Accuracy (on original test data):** 0.9232

**Performance Improvement:** The test set accuracy for LightGBM improved slightly from **0.9207** (pre-tuned) to **0.9232** (tuned), indicating a modest gain in generalization ability after optimizing its hyperparameters.

### Tuned CatBoost (without SMOTE)

For the CatBoost model, tuned on the original training data without SMOTE, `RandomizedSearchCV` yielded these best parameters:

*   **Best Parameters:** `{'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 500, 'depth': 10, 'border_count': 128}`
*   **Best Cross-Validation Accuracy (on original training data):** 0.9316
*   **Test Set Accuracy (on original test data):** 0.9277

**Performance Improvement:** The test set accuracy for CatBoost improved from **0.9229** (pre-tuned) to **0.9277** (tuned), showing a beneficial impact of hyperparameter optimization.

### Overall Conclusion

Comparing the two best-performing and tuned models on the test set:

*   **Tuned LightGBM (with SMOTE):** 0.9232 Accuracy
*   **Tuned CatBoost (without SMOTE):** 0.9277 Accuracy

**The Tuned CatBoost Classifier (without SMOTE) performed marginally better on the original test set, achieving an accuracy of 0.9277 compared to the Tuned LightGBM (with SMOTE) at 0.9232.** This suggests that for this particular dataset, directly addressing class imbalance through `auto_class_weights` in CatBoost and then tuning, proved slightly more effective for generalization to the original test set than using SMOTE with LightGBM, despite SMOTE leading to higher cross-validation scores on the balanced training data. Both models showed improved performance after hyperparameter tuning.

In [ ]:
import pandas as pd

# Data for the comparison table
comparison_data = {
    'Model': ['Tuned LightGBM (with SMOTE)', 'Tuned CatBoost (without SMOTE)'],
    'Train Accuracy (CV Mean)': [0.9612, 0.9316],
    'Test Accuracy': [0.9232, 0.9277],
    'F1 Score (Macro Avg)': [0.93, 0.94],
    'Overfitting (Y/N)': ['Y', 'N']
}

# Create the DataFrame
model_comparison_df = pd.DataFrame(comparison_data)

print("\n--- Model Comparison Table ---")
display(model_comparison_df)

In [ ]:
import pandas as pd

# Add a source column to differentiate results
results_df['Data Treatment'] = 'Original'
smote_results_df['Data Treatment'] = 'SMOTE'

# Rename columns for clarity in combined table if needed
results_df_display = results_df[['Model', 'Data Treatment', 'Accuracy (Mean)', 'F1-Score (Mean)']]
smote_results_df_display = smote_results_df[['Model', 'Data Treatment', 'Accuracy (Mean)', 'F1-Score (Mean)']]

# Combine the dataframes
all_models_comparison = pd.concat([results_df_display, smote_results_df_display], ignore_index=True)

# Sort by Accuracy (Mean) in descending order
all_models_comparison = all_models_comparison.sort_values(by='Accuracy (Mean)', ascending=False).reset_index(drop=True)

print("\n--- Comprehensive Model Comparison Table (Cross-Validation Results) ---")
display(all_models_comparison)

This table provides a comprehensive overview of all models evaluated, both with and without SMOTE, based on their cross-validation performance. It allows us to see how each model performed under different data treatment strategies.

In [ ]:
import pandas as pd

# Create a copy for the final comparison table
final_comparison_df = all_models_comparison.copy()

# Rename columns for clarity as per user request for cross-validation metrics
final_comparison_df = final_comparison_df.rename(columns={
    'Accuracy (Mean)': 'Train Accuracy (CV Mean)',
    'F1-Score (Mean)': 'F1 Score (CV Mean)'
})

# Add placeholder columns for Test Accuracy, Test F1 Score, and Overfitting
final_comparison_df['Test Accuracy'] = None
final_comparison_df['Test F1 Score (Macro Avg)'] = None
final_comparison_df['Overfitting (Y/N)'] = None

# Populate the new columns from model_comparison_df for the tuned models
for index, row in model_comparison_df.iterrows():
    tuned_model_name_full = row['Model']
    test_accuracy = row['Test Accuracy']
    test_f1_score = row['F1 Score (Macro Avg)']
    overfitting = row['Overfitting (Y/N)']

    # Extract base model name and data treatment from the tuned model name for matching
    base_model = ''
    data_treatment = ''
    if 'Tuned LightGBM (with SMOTE)' == tuned_model_name_full:
        base_model = 'LightGBM'
        data_treatment = 'SMOTE'
    elif 'Tuned CatBoost (without SMOTE)' == tuned_model_name_full:
        base_model = 'CatBoost'
        data_treatment = 'Original'
    else:
        continue # Skip if model name doesn't match expected tuned models

    # Find the corresponding row in the final_comparison_df using .loc
    match_indices = final_comparison_df.loc[
        (final_comparison_df['Model'] == base_model) &
        (final_comparison_df['Data Treatment'] == data_treatment)
    ].index

    if not match_indices.empty:
        final_comparison_df.loc[match_indices, 'Test Accuracy'] = test_accuracy
        final_comparison_df.loc[match_indices, 'Test F1 Score (Macro Avg)'] = test_f1_score
        final_comparison_df.loc[match_indices, 'Overfitting (Y/N)'] = overfitting

# Display the final table
print("\n--- Comprehensive Model Comparison with Test Set Metrics (All Classifiers) ---")
display(final_comparison_df)


Final Summary: Hyperparameter Tuning, Accuracy, and Production Recommendation
Our analysis focused on optimizing the two top-performing models: LightGBM (with SMOTE-resampled data) and CatBoost (with original data), both utilizing RandomizedSearchCV for hyperparameter tuning.

1. Hyperparameter Tuning Analysis:
Tuned LightGBM (with SMOTE):

Best Parameters: {'subsample': 0.8, 'reg_lambda': 0.1, 'reg_alpha': 0, 'num_leaves': 31, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 12, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Tuning led to a slight improvement in test set accuracy from 0.9207 to 0.9232. The num_leaves and n_estimators were optimized, along with regularization (reg_alpha, reg_lambda) and learning rate, to refine the model's complexity and learning steps.
Observation: While the cross-validation accuracy on the SMOTE-resampled training data was very high (0.9612), there was a noticeable drop to the test set accuracy (0.9232), indicating a degree of overfitting to the synthetic samples generated by SMOTE.
Tuned CatBoost (without SMOTE):

Best Parameters: {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 500, 'depth': 10, 'border_count': 128}
Tuning improved the test set accuracy from 0.9229 to 0.9277. Key parameters like iterations (number of trees), learning_rate, depth, and l2_leaf_reg (L2 regularization) were optimized to control model complexity and prevent overfitting.
Observation: This model showed a very consistent performance between its cross-validation accuracy (0.9316) and its test set accuracy (0.9277), suggesting strong generalization and minimal overfitting.
2. Model with Better Accuracy:
Based on the performance on the unseen test set, the Tuned CatBoost Classifier (without SMOTE) achieved a marginally better accuracy of 0.9277 compared to the Tuned LightGBM (with SMOTE) which achieved 0.9232.

3. Production Recommendation:
For production deployment, the Tuned CatBoost Classifier (without SMOTE) is the recommended model.

Reasons for Recommendation:

Higher Test Set Accuracy: It provided the best overall accuracy on the real, unseen test data.
Robustness to Overfitting: The minimal difference between its cross-validation accuracy (0.9316) and test set accuracy (0.9277), as well as the 'Overfitting (N)' flag, indicates excellent generalization capabilities. This suggests the model is less likely to perform poorly on new, real-world data.
Simplicity in Preprocessing: By not requiring SMOTE, the deployment pipeline for this model might be slightly simpler, as it avoids the step of generating synthetic data, which can sometimes introduce complexities or slight biases, as observed with the LightGBM model's performance drop.
Handling Imbalance: CatBoost's auto_class_weights effectively handled the class imbalance during its initial training, leading to robust performance even without explicit oversampling like SMOTE.
While SMOTE generally helps improve recall for minority classes, in this specific case, the CatBoost model's inherent robustness and slightly superior generalization on the original test set make it the preferred choice for a production environment where consistent performance on real-world data is paramount.